# Ensemble of Four Models for Pen Classification
Ensembling ConvNeXt Tiny, EfficientNet B4, ResNet50, and Swin Tiny models

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from tqdm import tqdm

## Load Test Data

In [ ]:
# -----------------------
# Load test CSV
# -----------------------
test_df = pd.read_csv("../test.csv")

# -----------------------
# Dataset (no labels)
# -----------------------
class TestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, row["image_id"]

# -----------------------
# Transform (same as val)
# -----------------------
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -----------------------
# DataLoader
# -----------------------
test_dataset = TestDataset(test_df, "", test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=8)

## Setup Device and Load Models

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------
# Load all four models
# -----------------------
models_list = []

# 1. ConvNeXt Tiny
model1 = models.convnext_tiny(weights="ConvNeXt_Tiny_Weights.DEFAULT")
in_features = model1.classifier[2].in_features
model1.classifier = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)
model1.load_state_dict(torch.load("best_model_convnext_tiny.pth", map_location=device))
model1 = model1.to(device)
model1.eval()
models_list.append(model1)
print("✅ ConvNeXt Tiny loaded")

# 2. EfficientNet B4
model2 = models.efficientnet_b4(weights="EfficientNet_B4_Weights.DEFAULT")
in_features = model2.classifier[1].in_features
model2.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)
model2.load_state_dict(torch.load("best_model_efficientnet_b4.pth", map_location=device))
model2 = model2.to(device)
model2.eval()
models_list.append(model2)
print("✅ EfficientNet B4 loaded")

# 3. ResNet50
model3 = models.resnet50(weights="ResNet50_Weights.DEFAULT")
in_features = model3.fc.in_features
model3.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)
model3.load_state_dict(torch.load("best_model_resnet50.pth", map_location=device))
model3 = model3.to(device)
model3.eval()
models_list.append(model3)
print("✅ ResNet50 loaded")

# 4. Swin Tiny
model4 = models.swin_t(weights="Swin_T_Weights.DEFAULT")
in_features = model4.head.in_features
model4.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)
model4.load_state_dict(torch.load("best_model_swin_tiny.pth", map_location=device))
model4 = model4.to(device)
model4.eval()
models_list.append(model4)
print("✅ Swin Tiny loaded")

print(f"Loaded {len(models_list)} models for ensembling")

Using device: cuda
✅ ConvNeXt Tiny loaded
✅ EfficientNet B4 loaded
✅ ResNet50 loaded
✅ Swin Tiny loaded
Loaded 4 models for ensembling


## Ensemble Inference with TTA

In [4]:
# -----------------------
# Ensemble Inference with TTA
# -----------------------
predictions = []
image_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(device)

        # Collect logits from all models for each augmentation
        all_logits = []

        # Original images
        for model in models_list:
            logits = model(images)
            all_logits.append(logits)

        # Horizontal flip
        flipped_images = torch.flip(images, dims=[3])
        for model in models_list:
            logits = model(flipped_images)
            all_logits.append(logits)

        # Vertical flip
        flipped_v_images = torch.flip(images, dims=[2])
        for model in models_list:
            logits = model(flipped_v_images)
            all_logits.append(logits)

        # Average all logits (12 total: 4 models × 3 augmentations)
        ensemble_logits = torch.stack(all_logits, dim=0).mean(dim=0)

        # Get predictions
        preds = torch.argmax(ensemble_logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        image_ids.extend(ids)

# -----------------------
# Convert back to 1-based labels
# -----------------------
predictions = [p + 1 for p in predictions]

# -----------------------
# Create submission
# -----------------------
submission = pd.DataFrame({
    "image_id": image_ids,
    "pen_id": predictions
})

submission.to_csv("submission_ensemble.csv", index=False)

print("✅ submission_ensemble.csv saved")
print(f"Total predictions: {len(predictions)}")
print(f"Unique predictions: {len(set(predictions))}")

100%|██████████| 185/185 [02:38<00:00,  1.16it/s]

✅ submission_ensemble.csv saved
Total predictions: 5905
Unique predictions: 8


## Check Prediction Distribution

In [ ]:
import matplotlib.pyplot as plt

# Check distribution of predictions
pred_counts = pd.Series(predictions).value_counts().sort_index()
print("Prediction distribution:")
print(pred_counts)

# Plot
plt.figure(figsize=(8, 5))
pred_counts.plot(kind='bar')
plt.xlabel('Pen ID')
plt.ylabel('Count')
plt.title('Ensemble Predictions Distribution')
plt.xticks(rotation=0)
plt.show()